<a href="https://colab.research.google.com/github/mdsadaqathali/week9/blob/main/w9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import GPT2Tokenizer,GPT2LMHeadModel,Trainer,TrainingArguments,DataCollatorForLanguageModeling
from datasets import load_dataset
import torch
import math

model_name="gpt2"

tokenizer=GPT2Tokenizer.from_pretrained(model_name)
model=GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token=tokenizer.eos_token

print("GPT-2 loaded successfully!")

dataset=load_dataset("daily_dialog")

print("Dataset loaded successfully!")
print(dataset)

def format_dialogue(example):
    text=""
    for dialogue in example["dialog"]:
        text+="SPEAKER_A: "+dialogue+" "
    return {"text":text}

dataset=dataset.map(format_dialogue)

train_dataset=dataset["train"]
test_dataset=dataset["validation"]

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train=train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test=test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

data_collator=DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args=TrainingArguments(
    output_dir="./gpt2-chatbot",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=500,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

print("\nStarting GPT-2 Fine-Tuning...")

trainer.train()

trainer.save_model("./gpt2-chatbot")
tokenizer.save_pretrained("./gpt2-chatbot")

print("\nFine-Tuning Completed!")

def generate_reply(prompt,max_length=100):
    inputs=tokenizer(prompt,return_tensors="pt")
    inputs={key:value.to(model.device) for key,value in inputs.items()}

    output=model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0],skip_special_tokens=True)

prompts=[
    "Hello, how are you?",
    "What do you like to do?",
    "Can you help me?",
    "What is your favorite movie?",
    "Tell me something interesting."
]

print("\nGenerated Responses:")

for prompt in prompts:
    print("\nPrompt:",prompt)
    print("Reply:",generate_reply(prompt))

print("\nChatbot Started!")
print("Type 'exit' to stop.")

history=""

while True:
    user_input=input("\nYou: ")

    if user_input.lower()=="exit":
        break

    history+=" SPEAKER_A: "+user_input

    reply=generate_reply(history,max_length=100)

    print("Bot:",reply)

    history+=" SPEAKER_B: "+reply

evaluation=trainer.evaluate()

eval_loss=evaluation["eval_loss"]
perplexity=math.exp(eval_loss)

print("\nEvaluation:")
print("Evaluation Loss:",round(eval_loss,4))
print("Perplexity:",round(perplexity,4))

print("\nProject Completed Successfully!")